# Phi-2 SAE + Field View (exp_001_sae_phi2)

Minimal Colab-runbook: tränar SAE på microsoft/phi-2 (layer 16) och kör tre field_view-logger (deterministisk / resonemang / hallucination) med vår risk-signal.

Kräver GPU-runtime i Colab.

In [1]:
# Sanity: GPU?\n
!nvidia-smi

/bin/bash: rad 1: nvidia-smi: kommandot finns inte


In [2]:
# Install deps\n
!pip install -q transformers accelerate torch

In [4]:
# Mount or clone repo (Colab)
import os, pathlib, subprocess

ROOT = pathlib.Path('Mechanistic Interpretability')
if not ROOT.exists():
    repo_url = os.environ.get('MI_REPO_URL', 'https://github.com/base76-research-lab/Mechanistic-Interpretability.git')
    try:
        print(f'Cloning {repo_url} → {ROOT}')
        subprocess.check_call(['git', 'clone', repo_url, str(ROOT)])
    except Exception as e:
        raise SystemExit("Upload the project folder or mount drive first. Set MI_REPO_URL to clone automatically.") from e

os.chdir(ROOT)
print('CWD', os.getcwd())



Cloning https://github.com/base76-research-lab/Mechanistic-Interpretability.git → Mechanistic Interpretability


Klonar till ”Mechanistic Interpretability”...
remote: Repository not found.
fatal: arkivet ”https://github.com/base76-research-lab/Mechanistic-Interpretability.git/” hittades inte


SystemExit: Upload the project folder or mount drive first. Set MI_REPO_URL to clone automatically.

/home/bjorn/.venvs/cognos/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Train SAE for Phi-2 (layer 16)
!python3 scripts/run_sae.py \
  --model microsoft/phi-2 \
  --layer 16 \
  --dict-size 512 \
  --steps 200 \
  --lr 1e-3 --l1 1e-3 \
  --layernorm \
  --suffix _phi2 \
  --device cuda

In [ ]:
# Run field_view with logging for three scenarios
COMMON_ARGS = "--model microsoft/phi-2 --layer 16 --sae_state experiments/exp_001_sae_phi2/sae_weights.pt --mode pc2 --topk 10 --device cuda"

!python3 scripts/run_field_view_logged.py --scenario math_det_phi2 --prompt "2 + 2 =" {COMMON_ARGS}
!python3 scripts/run_field_view_logged.py --scenario analogy_reason_phi2 --prompt "king is to queen as man is to" {COMMON_ARGS}
!python3 scripts/run_field_view_logged.py --scenario hallucination_phi2 --prompt "who was the president of france in 1200?" {COMMON_ARGS}

In [ ]:
# List artifacts and run logs
!ls experiments/exp_001_sae_phi2/artifacts
!ls experiments/exp_001_sae_phi2/runs

Kopiera hem:
```
zip -r phi2_results.zip experiments/exp_001_sae_phi2
```
Ladda ner `phi2_results.zip` till din maskin och lägg under projektet.